# PS02 — Kaggle GPU Diffusion Inference Notebook

> **Environment:** Kaggle Notebooks (Free GPU — NVIDIA T4, 16GB VRAM)
> **Model:** `stabilityai/stable-diffusion-xl-base-1.0` (Pinned Revision: `462165984030d82259a11f4367a4eed129e94a7b`)
> **VAE:** `madebyollin/sdxl-vae-fp16-fix` (Pinned Revision: `207b116dae70ace3637169f1ddd2434b91b3a8cd`)
> **Compliance:** Free accelerator only. Google Colab is strictly forbidden. Zero commercial/paid APIs.

In [ ]:
# Cell 1: Install dependencies with compatible accelerate and numpy < 2
!pip install --quiet "numpy<2.0.0" diffusers==0.29.2 transformers==4.42.3 --upgrade accelerate safetensors==0.4.3 torchvision

In [ ]:
# Cell 2: System and GPU verification
import sys
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA not detected! In Kaggle, go to Settings -> Accelerator -> select GPU T4.")

In [ ]:
# Cell 3: Job specification definition and loading
import os
import json
from datetime import datetime, timezone
from pathlib import Path

# Locate uploaded job bundle or use default baseline avatar spec
job_path_env = os.environ.get("KAGGLE_JOB_PATH", "/kaggle/input/avatar-job/job.json")

default_job = {
    "job_id": "kaggle-demo-run-001",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model": {
        "name": "stable-diffusion-xl-base-1.0",
        "repo": "stabilityai/stable-diffusion-xl-base-1.0",
        "revision": "462165984030d82259a11f4367a4eed129e94a7b",
        "vae_repo": "madebyollin/sdxl-vae-fp16-fix",
        "vae_revision": "207b116dae70ace3637169f1ddd2434b91b3a8cd"
    },
    "prompt": "a feminine young adult person, Fitzpatrick III (warm olive) skin tone, shoulder-length wavy chestnut brown hair, wearing navy tailored business blazer, crisp white shirt, confident front-facing portrait pose, modern sunlit creative studio, soft warm bokeh, professional portrait photography, sharp focus, high resolution, 8k, photorealistic, studio lighting",
    "negative_prompt": "nsfw, explicit, nudity, violence, gore, ugly, deformed, blurry, low quality, watermark, signature, text, logo, duplicate, mutation, bad anatomy, extra limbs, cartoon, anime, illustration, painting",
    "seed": 424242,
    "inference_params": {
        "steps": 20,
        "guidance_scale": 7.5,
        "aspect_ratio": "1:1",
        "output_width": 1024
    },
    "provenance_route": "kaggle"
}

if os.path.exists(job_path_env):
    print(f"Loading custom job bundle from: {job_path_env}")
    with open(job_path_env, "r", encoding="utf-8") as f:
        job = json.load(f)
else:
    print("Using exemplar baseline job bundle (young adult feminine avatar).")
    job = default_job

print(f"Job ID: {job['job_id']}")
print(f"Seed: {job['seed']}")
print(f"Prompt: {job['prompt'][:85]}...")

In [ ]:
# Compatibility shim: bypass Kaggle peft/accelerate version mismatch
import torch
import accelerate.utils.memory
accelerate.utils.memory.clear_device_cache = lambda: torch.cuda.empty_cache() if torch.cuda.is_available() else None
try:
    import diffusers.pipelines.pipeline_loading_utils
    diffusers.pipelines.pipeline_loading_utils.is_peft_available = lambda: False
except Exception:
    pass

# Cell 4: Load pinned SDXL pipeline with FP16 VAE
import torch
from diffusers import StableDiffusionXLPipeline, AutoencoderKL

model_repo = job["model"].get("repo", "stabilityai/stable-diffusion-xl-base-1.0")
model_rev = job["model"].get("revision", "462165984030d82259a11f4367a4eed129e94a7b")
vae_repo = job["model"].get("vae_repo", "madebyollin/sdxl-vae-fp16-fix")
vae_rev = job["model"].get("vae_revision", "207b116dae70ace3637169f1ddd2434b91b3a8cd")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# Load VAE with graceful fallback
vae = None
try:
    print(f"Loading VAE: {vae_repo} (rev: {vae_rev[:10]})...")
    vae = AutoencoderKL.from_pretrained(
        vae_repo,
        revision=vae_rev,
        torch_dtype=dtype
    )
    print("FP16 VAE loaded successfully.")
except Exception as e:
    print(f"Notice: Using built-in SDXL VAE ({e})")
    vae = None

print(f"Loading SDXL Pipeline: {model_repo} (rev: {model_rev[:10]})...")
pipe_kwargs = {
    "revision": model_rev,
    "torch_dtype": dtype,
    "use_safetensors": True,
    "variant": "fp16" if torch.cuda.is_available() else None
}
if vae is not None:
    pipe_kwargs["vae"] = vae

pipe = StableDiffusionXLPipeline.from_pretrained(
    model_repo,
    **pipe_kwargs
)

if torch.cuda.is_available():
    pipe.enable_attention_slicing()
    pipe = pipe.to("cuda")

print("SDXL model successfully loaded on GPU.")

In [ ]:
# Neutralize Kaggle's incompatible peft package (not needed for base SDXL inference)
import sys
from types import ModuleType
mock_peft = ModuleType('peft')
mock_peft.PeftModel = type('PeftModel', (), {})
mock_peft.__version__ = '0.0.0'
mock_tuners = ModuleType('peft.tuners.tuners_utils')
mock_tuners.BaseTunerLayer = type('BaseTunerLayer', (), {})
sys.modules['peft'] = mock_peft
sys.modules['peft.tuners'] = ModuleType('peft.tuners')
sys.modules['peft.tuners.tuners_utils'] = mock_tuners
try:
    import diffusers.utils.peft_utils as pu
    pu.USE_PEFT_BACKEND = False
    pu.unscale_lora_layers = lambda *args, **kwargs: None
    pu.scale_lora_layers = lambda *args, **kwargs: None
    import diffusers.pipelines.stable_diffusion_xl.pipeline_stable_diffusion_xl as sdxl_mod
    sdxl_mod.USE_PEFT_BACKEND = False
except Exception:
    pass

# Cell 5: Deterministic diffusion inference and result fragment generation
import os
import json
import time
import hashlib
from datetime import datetime, timezone
from pathlib import Path
import torch

aspect = job["inference_params"].get("aspect_ratio", "1:1")
dim_map = {
    "1:1": (1024, 1024),
    "3:4": (896, 1152),
    "9:16": (768, 1344)
}
width, height = dim_map.get(aspect, (1024, 1024))
steps = job["inference_params"].get("steps", 20)
guidance = job["inference_params"].get("guidance_scale", 7.5)
seed = int(job["seed"])

gen_device = "cuda" if torch.cuda.is_available() else "cpu"
generator = torch.Generator(device=gen_device).manual_seed(seed)

output_dir = Path("/kaggle/working/output")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Generating avatar: {width}x{height}, {steps} steps, seed={seed}...")
t_start = time.perf_counter()

result = pipe(
    prompt=job["prompt"],
    negative_prompt=job["negative_prompt"],
    width=width,
    height=height,
    num_inference_steps=steps,
    guidance_scale=guidance,
    generator=generator
)
image = result.images[0]
t_elapsed = time.perf_counter() - t_start
print(f"Inference complete in {t_elapsed:.2f}s")

# Save image
img_filename = f"avatar_{job['job_id'][:8]}_{seed}.png"
img_path = output_dir / img_filename
image.save(img_path)

# Compute SHA-256 hash
h = hashlib.sha256()
with open(img_path, "rb") as f:
    for chunk in iter(lambda: f.read(65536), b""):
        h.update(chunk)
img_hash = h.hexdigest()

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

# Save result_fragment.json for avatarpipe ingest
fragment = {
    "job_id": job["job_id"],
    "images": [str(img_path)],
    "seed": seed,
    "model_id": job["model"]["name"],
    "revision": job["model"]["revision"],
    "provenance_route": "kaggle",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "inference_time_seconds": round(t_elapsed, 3),
    "gpu_device": gpu_name,
    "image_hashes": {
        img_filename: img_hash
    }
}

fragment_file = output_dir / "result_fragment.json"
with open(fragment_file, "w", encoding="utf-8") as f:
    json.dump(fragment, f, indent=2)

print(f"Saved image:    {img_path}")
print(f"Saved fragment: {fragment_file}")
print("Ready for downloading and local ingestion via avatarpipe ingest!")